# MITeNAM: Multi-Teacher Input-Injection Neural Additive Model
## UCI Diabetes 130-US Hospitals — Reproducible Pipeline

This notebook reproduces the experimental results reported in our CIKM 2026 paper
for the UCI Diabetes 30-day readmission prediction task.

**Folder structure (Google Drive):**
```
mitenam_cikm26/
├── data/
│   └── diabetic_data.csv           ← UCI dataset (place here)
├── mitenam_uci_pipeline.ipynb      ← this notebook
└── results/                        ← auto-generated
    ├── uci_xgb.pkl
    ├── uci_cat.pkl
    ├── uci_lgb.pkl
    ├── uci_ebm.pkl
    └── uci_mitenam.pkl
```

**Reproducibility:** All random seeds, hyperparameters, preprocessing steps, and
evaluation procedures match the configuration that produced the paper's Table 3.

**Runtime:** Approximately 1.5 hours on a T4 GPU, 40–50 minutes on an A100 GPU.


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

BASE = '/content/drive/MyDrive/mitenam_cikm26/'
print(f"Working directory: {BASE}")

## 2. Install Packages

In [ ]:
!pip install catboost lightgbm interpret -q

## 3. Setup: Imports, Constants, Evaluation Utilities

**Statistical test:** Real DeLong's test (Sun & Xu, 2014, fast implementation of DeLong 1988)
for comparing two correlated AUROC values.

**F1 score:** Computed at the threshold that maximizes F1 on the out-of-fold predictions,
following the protocol described in the paper (`F1 threshold is chosen on the validation
fold to maximize F1`).


In [ ]:
import os
import pickle
import time
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from scipy import stats
import torch
import torch.nn as nn

warnings.filterwarnings('ignore')

# ── Paths ──
DATA_PATH = BASE + 'data/diabetic_data.csv'
RESULTS_DIR = BASE + 'results/'
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Reproducibility constants ──
SEEDS = [42, 0, 2026]      # 3 outer CV seeds
N_FOLDS = 5                # 5-fold stratified CV
NAM_SEEDS = [42, 1234, 5678]  # 3 NAM ensemble seeds

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


# ──────────────────────────────────────────────────────────
# F1 with optimal threshold (chosen to maximize F1 on OOF)
# ──────────────────────────────────────────────────────────
def best_f1_threshold(y_true, p, n_grid=399):
    """Find the decision threshold that maximizes F1 on the given OOF predictions.

    Returns
    -------
    best_f1 : float
        Maximum F1 score over the candidate threshold grid.
    best_threshold : float
        Threshold achieving best_f1.
    """
    qs = np.linspace(0.005, 0.995, n_grid)
    candidates = np.unique(np.quantile(p, qs))
    best_f1, best_threshold = -1.0, 0.5
    for t in candidates:
        f1 = f1_score(y_true, (p >= t).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_threshold = f1, t
    return best_f1, best_threshold


# ──────────────────────────────────────────────────────────
# Evaluation — AUROC, AUPRC, and F1 (optimal threshold)
# ──────────────────────────────────────────────────────────
def evaluate(y_true, y_proba):
    """Compute AUROC, AUPRC, and F1 (at the threshold maximizing F1)."""
    f1, _ = best_f1_threshold(y_true, y_proba)
    return {
        'AUROC': roc_auc_score(y_true, y_proba),
        'AUPRC': average_precision_score(y_true, y_proba),
        'F1':    f1,
    }


# ──────────────────────────────────────────────────────────
# Real DeLong's test (Sun & Xu 2014, fast implementation)
# ──────────────────────────────────────────────────────────
def fast_delong(y_true, p_a, p_b):
    """DeLong's test for two correlated AUROC values.

    Based on Sun & Xu (2014), the fast implementation of DeLong (1988).

    Returns
    -------
    auc_a : float
        AUROC of model A (matches sklearn.metrics.roc_auc_score).
    auc_b : float
        AUROC of model B.
    p_value : float
        Two-sided p-value for H0: AUC_a = AUC_b.
    """
    y_true = np.asarray(y_true).astype(int)
    p_a = np.asarray(p_a).astype(float)
    p_b = np.asarray(p_b).astype(float)

    def midrank(x):
        J = np.argsort(x, kind='mergesort')
        Z = x[J]
        N = len(x)
        T = np.zeros(N)
        i = 0
        while i < N:
            j = i
            while j < N and Z[j] == Z[i]:
                j += 1
            T[i:j] = 0.5 * (i + j - 1) + 1
            i = j
        out = np.empty(N)
        out[J] = T
        return out

    pos_idx = np.where(y_true == 1)[0]
    neg_idx = np.where(y_true == 0)[0]
    m = len(pos_idx)
    n = len(neg_idx)

    preds = np.vstack([p_a, p_b])
    reordered = np.concatenate([pos_idx, neg_idx])
    preds_r = preds[:, reordered]

    tx = np.empty((2, m))
    ty = np.empty((2, n))
    tz = np.empty((2, m + n))
    for r in range(2):
        tx[r] = midrank(preds_r[r, :m])
        ty[r] = midrank(preds_r[r, m:])
        tz[r] = midrank(preds_r[r])

    aucs = tz[:, :m].sum(axis=1) / m / n - (m + 1) / (2 * n)

    v01 = (tz[:, :m] - tx) / n
    v10 = 1 - (tz[:, m:] - ty) / m

    sx = np.cov(v01)
    sy = np.cov(v10)

    delta = np.array([1, -1])
    var = (delta @ sx @ delta) / m + (delta @ sy @ delta) / n

    if var <= 0:
        return aucs[0], aucs[1], 1.0

    z = (aucs[0] - aucs[1]) / np.sqrt(var)
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))
    return aucs[0], aucs[1], p_value


# ──────────────────────────────────────────────────────────
# Save / load
# ──────────────────────────────────────────────────────────
def save_result(name, result):
    path = os.path.join(RESULTS_DIR, f'{name}.pkl')
    with open(path, 'wb') as f:
        pickle.dump(result, f)
    print(f"💾 Saved: {path}")


def load_result(name):
    path = os.path.join(RESULTS_DIR, f'{name}.pkl')
    if not os.path.exists(path):
        return None
    with open(path, 'rb') as f:
        return pickle.load(f)


print("✅ Setup complete.")

## 4. Load and Preprocess UCI Diabetes Dataset

**Cohort filtering:** Encounters with discharge disposition recorded as expired or hospice
(codes 11, 13, 14, 19, 20, 21) are excluded, yielding **99,343 encounters** with a 30-day
readmission positive rate of **11.39%**.

**Feature engineering:**
- ICD-9 codes (`diag_1`, `diag_2`, `diag_3`) are mapped to 9 broad categories
  (Circulatory, Respiratory, Digestive, Diabetes, Injury, Musculoskeletal,
  Genitourinary, Neoplasms, Other).
- Age ranges (e.g., `[0-10)`) are converted to the midpoint integer.
- Identifier columns and columns with excessive missingness are dropped.
- Numerical features: median imputation; categorical features: missing values filled with `'Missing'`.


In [ ]:
# ── Load ──
df_uci = pd.read_csv(DATA_PATH)
print(f"UCI raw: {df_uci.shape}")

# ── 1. Target: 30-day readmission (<30 days = 1) ──
df_uci['readmit_30d'] = (df_uci['readmitted'] == '<30').astype(int)

# ── 2. Replace '?' with NaN ──
df_uci = df_uci.replace('?', np.nan)

# ── 3. Drop irrelevant columns ──
drop_cols = [
    'encounter_id', 'patient_nbr',
    'weight',             # >97% missing
    'payer_code',         # high missingness, irrelevant
    'medical_specialty',  # high missingness
    'examide', 'citoglipton',  # constant 'No'
    'readmitted',         # raw target column (replaced)
]
df_uci = df_uci.drop(columns=[c for c in drop_cols if c in df_uci.columns])

# ── 4. Exclude expired/hospice (discharge_disposition_id ∈ {11, 13, 14, 19, 20, 21}) ──
expire_codes = [11, 13, 14, 19, 20, 21]
df_uci = df_uci[~df_uci['discharge_disposition_id'].isin(expire_codes)].reset_index(drop=True)

# ── 5. Age band → midpoint integer ──
def age_to_num(s):
    if pd.isna(s):
        return np.nan
    return int(s.replace('[', '').split('-')[0]) + 5

df_uci['age'] = df_uci['age'].apply(age_to_num)

# ── 6. ICD-9 codes → 9 broad categories ──
def icd_to_category(c):
    if pd.isna(c):
        return 'Other'
    try:
        c_str = str(c)
        if c_str.startswith('V') or c_str.startswith('E'):
            return 'Other'
        c = float(c_str)
        if 390 <= c < 460 or c == 785: return 'Circulatory'
        if 460 <= c < 520 or c == 786: return 'Respiratory'
        if 520 <= c < 580 or c == 787: return 'Digestive'
        if c == 250 or (249 < c < 251): return 'Diabetes'
        if 800 <= c < 1000: return 'Injury'
        if 710 <= c < 740: return 'Musculoskeletal'
        if 580 <= c < 630 or c == 788: return 'Genitourinary'
        if 140 <= c < 240: return 'Neoplasms'
        return 'Other'
    except:
        return 'Other'

for c in ['diag_1', 'diag_2', 'diag_3']:
    df_uci[f'{c}_cat'] = df_uci[c].apply(icd_to_category)
    df_uci = df_uci.drop(columns=[c])

# ── 7. Features and target ──
TARGET_UCI = 'readmit_30d'
y_uci = df_uci[TARGET_UCI].values
X_uci_full = df_uci.drop(columns=[TARGET_UCI])

# ── 8. Auto-classify numerical / categorical ──
CAT_COLS_UCI = []
NUM_COLS_UCI = []
for c in X_uci_full.columns:
    if X_uci_full[c].dtype == 'object' or X_uci_full[c].nunique() <= 10:
        CAT_COLS_UCI.append(c)
    else:
        NUM_COLS_UCI.append(c)

# ── 9. Impute missing values ──
for c in NUM_COLS_UCI:
    X_uci_full[c] = pd.to_numeric(X_uci_full[c], errors='coerce')
    X_uci_full[c] = X_uci_full[c].fillna(X_uci_full[c].median())

for c in CAT_COLS_UCI:
    X_uci_full[c] = X_uci_full[c].fillna('Missing').astype(str)

print(f"\n✅ UCI preprocessing complete")
print(f"   Sample size: {len(df_uci)}")
print(f"   Positive rate: {y_uci.mean():.4f}")
print(f"   Numerical features: {len(NUM_COLS_UCI)}")
print(f"   Categorical features: {len(CAT_COLS_UCI)}")

## 5. NAM Student Model + Training Function

The NAM student decomposes prediction into independent per-feature shape functions.
Each numerical feature and each categorical feature (after a 1-D learnable embedding)
is processed by an independent 3-layer MLP subnetwork; outputs are additively combined.

**Training hyperparameters** (matching the paper's reported configuration):
- Adam optimizer, lr=1e-3, weight_decay=1e-4
- batch_size=2048, max_epochs=50, patience=10
- T=2.0, alpha=0.5
- L2 ridge regularization coefficient = 0.01


In [ ]:
# ── NAM subnetwork: 1 → 64 → 64 → 32 → 1 with ReLU and dropout ──
class NAMSubnet(nn.Module):
    def __init__(self, hidden=(64, 64, 32), dropout=0.1):
        super().__init__()
        layers = []
        in_dim = 1
        for h in hidden:
            layers += [nn.Linear(in_dim, h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


# ── NAM student: numerical + categorical with 1-D embedding ──
class NAMStudent(nn.Module):
    def __init__(self, n_num, cat_dims, hidden=(64, 64, 32), dropout=0.1, feat_drop=0.05):
        super().__init__()
        self.feat_drop = feat_drop
        self.num_subnets = nn.ModuleList([NAMSubnet(hidden, dropout) for _ in range(n_num)])
        self.cat_embeds = nn.ModuleList([nn.Embedding(n, 1) for n in cat_dims])
        self.cat_subnets = nn.ModuleList([NAMSubnet(hidden, dropout) for _ in cat_dims])
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, x_num, x_cat):
        outs = [s(x_num[:, i:i+1]) for i, s in enumerate(self.num_subnets)]
        for i, (emb, s) in enumerate(zip(self.cat_embeds, self.cat_subnets)):
            outs.append(s(emb(x_cat[:, i])))
        if self.training and self.feat_drop > 0:
            for i in range(len(outs)):
                if torch.rand(1).item() < self.feat_drop:
                    outs[i] = outs[i] * 0
        logit = torch.stack(outs, dim=-1).sum(dim=-1) + self.bias
        return logit


# ── NAM training with KD loss + early stopping ──
def train_nam(X_tr_num, X_tr_cat, y_tr, X_val_num, X_val_cat, y_val,
              teacher_logit_tr, cat_dims, seed, device, max_epochs=50, patience=10):
    torch.manual_seed(seed)
    np.random.seed(seed)

    n_num = X_tr_num.shape[1]
    model = NAMStudent(n_num=n_num, cat_dims=cat_dims).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

    X_tr_num_t = torch.FloatTensor(X_tr_num).to(device)
    X_tr_cat_t = torch.LongTensor(X_tr_cat).to(device)
    y_tr_t = torch.FloatTensor(y_tr).to(device)
    teacher_logit_t = torch.FloatTensor(teacher_logit_tr).to(device)
    X_val_num_t = torch.FloatTensor(X_val_num).to(device)
    X_val_cat_t = torch.LongTensor(X_val_cat).to(device)

    T, alpha = 2.0, 0.5
    best_val_auc = 0
    best_state = None
    no_improve = 0
    bs = 2048
    n_train = len(y_tr)

    for epoch in range(max_epochs):
        model.train()
        perm = np.random.permutation(n_train)
        for i in range(0, n_train, bs):
            b_idx = perm[i:i+bs]
            b_num = X_tr_num_t[b_idx]
            b_cat = X_tr_cat_t[b_idx]
            b_y = y_tr_t[b_idx]
            b_tl = teacher_logit_t[b_idx]

            logit = model(b_num, b_cat)
            bce = nn.functional.binary_cross_entropy_with_logits(logit, b_y)
            student_soft = torch.sigmoid(logit / T)
            teacher_soft = torch.sigmoid(b_tl / T)
            kl = nn.functional.binary_cross_entropy(student_soft, teacher_soft.detach())

            loss = alpha * bce + (1 - alpha) * (T ** 2) * kl

            # NAM L2 ridge regularization
            reg = sum((p ** 2).mean() for p in model.parameters() if p.requires_grad) * 0.01
            loss = loss + reg

            opt.zero_grad()
            loss.backward()
            opt.step()

        # Validation AUROC for early stopping
        model.eval()
        with torch.no_grad():
            val_logit = model(X_val_num_t, X_val_cat_t)
            val_proba = torch.sigmoid(val_logit).cpu().numpy()
        val_auc = roc_auc_score(y_val, val_proba)

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    model.load_state_dict(best_state)
    return model


# ── Joint preprocessing: numerical (standardize) + categorical (label-encode + one-hot) ──
def prep_data(X_tr, X_te, num_cols, cat_cols):
    X_tr = X_tr.copy()
    X_te = X_te.copy()

    # Numerical: median-impute + standardize
    imp = SimpleImputer(strategy='median')
    X_tr_num = imp.fit_transform(X_tr[num_cols])
    X_te_num = imp.transform(X_te[num_cols])
    scaler = StandardScaler()
    X_tr_num = scaler.fit_transform(X_tr_num).astype(np.float32)
    X_te_num = scaler.transform(X_te_num).astype(np.float32)

    # Categorical: integer-encode for NAM embedding
    cat_dims = []
    X_tr_cat_list, X_te_cat_list = [], []
    for c in cat_cols:
        tr_v = X_tr[c].astype(str)
        te_v = X_te[c].astype(str)
        uniq = list(tr_v.unique())
        m = {v: i+1 for i, v in enumerate(uniq)}
        X_tr_cat_list.append(tr_v.map(m).fillna(0).astype(int).values)
        X_te_cat_list.append(te_v.map(lambda v: m.get(v, 0)).astype(int).values)
        cat_dims.append(len(uniq) + 1)

    if cat_cols:
        X_tr_cat = np.stack(X_tr_cat_list, axis=1).astype(np.int64)
        X_te_cat = np.stack(X_te_cat_list, axis=1).astype(np.int64)
    else:
        X_tr_cat = np.zeros((len(X_tr), 0), dtype=np.int64)
        X_te_cat = np.zeros((len(X_te), 0), dtype=np.int64)

    # Flat representation for GBDT teachers and EBM (one-hot for categorical)
    X_tr_oh = pd.get_dummies(X_tr[cat_cols], drop_first=False) if cat_cols else pd.DataFrame()
    X_te_oh = pd.get_dummies(X_te[cat_cols], drop_first=False) if cat_cols else pd.DataFrame()
    X_te_oh = X_te_oh.reindex(columns=X_tr_oh.columns, fill_value=0)
    X_tr_num_df = pd.DataFrame(X_tr_num, columns=num_cols, index=X_tr.index)
    X_te_num_df = pd.DataFrame(X_te_num, columns=num_cols, index=X_te.index)
    X_tr_flat = pd.concat([X_tr_num_df, X_tr_oh], axis=1).values.astype(np.float32)
    X_te_flat = pd.concat([X_te_num_df, X_te_oh], axis=1).values.astype(np.float32)

    return X_tr_num, X_te_num, X_tr_cat, X_te_cat, cat_dims, X_tr_flat, X_te_flat


# ── 3-seed × 5-fold runner ──
def run_3seed_5fold(X_full, y, num_cols, cat_cols, model_name, fit_predict_fn):
    """3 outer seeds × 5 stratified folds = 15 fold-level evaluations."""
    start = time.time()
    n = len(y)
    oof = np.zeros((n, len(SEEDS)))
    fold_aucs = []

    for s_idx, seed in enumerate(SEEDS):
        print(f"\n── Seed {seed} ──")
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        seed_oof = np.zeros(n)
        for fold, (tr, te) in enumerate(skf.split(X_full, y)):
            X_tr, X_te = X_full.iloc[tr], X_full.iloc[te]
            y_tr, y_te = y[tr], y[te]
            y_proba = fit_predict_fn(X_tr, y_tr, X_te, num_cols, cat_cols)
            seed_oof[te] = y_proba
            auc = roc_auc_score(y_te, y_proba)
            print(f"  Fold {fold+1}: AUROC={auc:.4f}")
            fold_aucs.append(auc)
        oof[:, s_idx] = seed_oof
        seed_auc = roc_auc_score(y, seed_oof)
        print(f"  → Seed {seed}: AUROC={seed_auc:.4f}")

    oof_avg = oof.mean(axis=1)
    final = evaluate(y, oof_avg)
    elapsed = time.time() - start
    print(f"\n📊 {model_name} FINAL: AUROC={final['AUROC']:.4f}  AUPRC={final['AUPRC']:.4f}  F1={final['F1']:.4f}  ⏱️ {elapsed/60:.1f}min")

    result = {
        'name': model_name,
        'oof': oof_avg,
        'oof_seeds': oof,
        'final': final,
        'fold_aucs': fold_aucs,
        'time': elapsed,
    }
    save_result(model_name, result)
    return result


print("✅ NAM student model and training utilities defined.")

## 6. Baselines + MITeNAM Training Functions

Each function takes `(X_tr, y_tr, X_te, num_cols, cat_cols)` and returns
test-fold predicted probabilities.

**Baseline hyperparameters** (consistent across teachers and baselines):
- GBDT models (XGBoost, CatBoost, LightGBM): `n_estimators=300`, `max_depth=6`,
  `learning_rate=0.05`, `scale_pos_weight=neg/pos` ratio, `random_state=42`
- EBM: default settings, `random_state=42`

**MITeNAM (`fit_mitenam`):**
- Inner train/val split: 15% stratified, seed 42
- 3 GBDT teachers trained, ensembled with cat_heavy weights:
  $(w_{xgb}, w_{cat}, w_{lgb}) = (0.25, 0.50, 0.25)$
- Teacher probability appended to numerical features
- NAM student ensemble (3 seeds: 42, 1234, 5678), sigmoid outputs averaged


In [ ]:
# ── XGBoost ──
def fit_xgb(X_tr, y_tr, X_te, num_cols, cat_cols):
    import xgboost as xgb
    _, _, _, _, _, X_tr_flat, X_te_flat = prep_data(X_tr, X_te, num_cols, cat_cols)
    spw = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)
    m = xgb.XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        scale_pos_weight=spw, tree_method='hist',
        random_state=42, n_jobs=-1, verbosity=0,
    )
    m.fit(X_tr_flat, y_tr)
    return m.predict_proba(X_te_flat)[:, 1]


# ── CatBoost ──
def fit_cat(X_tr, y_tr, X_te, num_cols, cat_cols):
    from catboost import CatBoostClassifier
    X_tr_p = X_tr.copy()
    X_te_p = X_te.copy()
    for c in cat_cols:
        X_tr_p[c] = X_tr_p[c].astype(str)
        X_te_p[c] = X_te_p[c].astype(str)
    cat_idx = [X_tr_p.columns.get_loc(c) for c in cat_cols]
    spw = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)
    m = CatBoostClassifier(
        iterations=300, depth=6, learning_rate=0.05,
        scale_pos_weight=spw, cat_features=cat_idx,
        random_state=42, verbose=False,
    )
    m.fit(X_tr_p, y_tr)
    return m.predict_proba(X_te_p)[:, 1]


# ── LightGBM ──
def fit_lgb(X_tr, y_tr, X_te, num_cols, cat_cols):
    import lightgbm as lgb
    _, _, _, _, _, X_tr_flat, X_te_flat = prep_data(X_tr, X_te, num_cols, cat_cols)
    spw = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)
    m = lgb.LGBMClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        scale_pos_weight=spw, objective='binary',
        random_state=42, n_jobs=-1, verbose=-1,
    )
    m.fit(X_tr_flat, y_tr)
    return m.predict_proba(X_te_flat)[:, 1]


# ── EBM ──
def fit_ebm(X_tr, y_tr, X_te, num_cols, cat_cols):
    from interpret.glassbox import ExplainableBoostingClassifier
    _, _, _, _, _, X_tr_flat, X_te_flat = prep_data(X_tr, X_te, num_cols, cat_cols)
    m = ExplainableBoostingClassifier(random_state=42)
    m.fit(X_tr_flat, y_tr)
    return m.predict_proba(X_te_flat)[:, 1]


# ── MITeNAM (Multi-Teacher Input-Injection Neural Additive Model) ──
def fit_mitenam(X_tr, y_tr, X_te, num_cols, cat_cols):
    import xgboost as xgb
    from catboost import CatBoostClassifier
    import lightgbm as lgb

    # Inner train/val split for early stopping (15% stratified)
    X_tr_in, X_val, y_tr_in, y_val = train_test_split(
        X_tr, y_tr, test_size=0.15, stratify=y_tr, random_state=42,
    )

    # Preprocessing
    X_tr_num, X_val_num, X_tr_cat, X_val_cat, cat_dims, X_tr_flat, X_val_flat = \
        prep_data(X_tr_in, X_val, num_cols, cat_cols)
    _, X_te_num, _, X_te_cat, _, _, X_te_flat = prep_data(X_tr_in, X_te, num_cols, cat_cols)

    spw = (y_tr_in == 0).sum() / max((y_tr_in == 1).sum(), 1)
    eps = 1e-7

    # ── Teacher 1: XGBoost ──
    t_xgb = xgb.XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        scale_pos_weight=spw, tree_method='hist',
        random_state=42, n_jobs=-1, verbosity=0,
    )
    t_xgb.fit(X_tr_flat, y_tr_in)
    p_xgb_tr = t_xgb.predict_proba(X_tr_flat)[:, 1]
    p_xgb_val = t_xgb.predict_proba(X_val_flat)[:, 1]
    p_xgb_te = t_xgb.predict_proba(X_te_flat)[:, 1]

    # ── Teacher 2: CatBoost ──
    t_cat = CatBoostClassifier(
        iterations=300, depth=6, learning_rate=0.05,
        scale_pos_weight=spw, random_state=42, verbose=False,
    )
    t_cat.fit(X_tr_flat, y_tr_in)
    p_cat_tr = t_cat.predict_proba(X_tr_flat)[:, 1]
    p_cat_val = t_cat.predict_proba(X_val_flat)[:, 1]
    p_cat_te = t_cat.predict_proba(X_te_flat)[:, 1]

    # ── Teacher 3: LightGBM ──
    t_lgb = lgb.LGBMClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        scale_pos_weight=spw, objective='binary',
        random_state=42, n_jobs=-1, verbose=-1,
    )
    t_lgb.fit(X_tr_flat, y_tr_in)
    p_lgb_tr = t_lgb.predict_proba(X_tr_flat)[:, 1]
    p_lgb_val = t_lgb.predict_proba(X_val_flat)[:, 1]
    p_lgb_te = t_lgb.predict_proba(X_te_flat)[:, 1]

    # ── Cat-heavy weighted ensemble ──
    w_xgb, w_cat, w_lgb = 0.25, 0.5, 0.25
    p_ens_tr = (w_xgb * p_xgb_tr + w_cat * p_cat_tr + w_lgb * p_lgb_tr).clip(eps, 1-eps)
    p_ens_val = (w_xgb * p_xgb_val + w_cat * p_cat_val + w_lgb * p_lgb_val).clip(eps, 1-eps)
    p_ens_te = (w_xgb * p_xgb_te + w_cat * p_cat_te + w_lgb * p_lgb_te).clip(eps, 1-eps)
    teacher_logit_tr = np.log(p_ens_tr / (1 - p_ens_tr)).astype(np.float32)

    # ── Append teacher probability to numerical features ──
    X_tr_num_aug = np.concatenate([X_tr_num, p_ens_tr.reshape(-1, 1).astype(np.float32)], axis=1)
    X_val_num_aug = np.concatenate([X_val_num, p_ens_val.reshape(-1, 1).astype(np.float32)], axis=1)
    X_te_num_aug = np.concatenate([X_te_num, p_ens_te.reshape(-1, 1).astype(np.float32)], axis=1)

    # ── NAM student ensemble (3 seeds) ──
    test_probas = []
    for s in NAM_SEEDS:
        model = train_nam(
            X_tr_num_aug, X_tr_cat, y_tr_in,
            X_val_num_aug, X_val_cat, y_val,
            teacher_logit_tr, cat_dims, s, device,
        )
        model.eval()
        with torch.no_grad():
            X_te_num_t = torch.FloatTensor(X_te_num_aug).to(device)
            X_te_cat_t = torch.LongTensor(X_te_cat).to(device)
            logit = model(X_te_num_t, X_te_cat_t)
            test_probas.append(torch.sigmoid(logit).cpu().numpy())
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    final_proba = np.mean(test_probas, axis=0)
    del t_xgb, t_cat, t_lgb
    return final_proba


print("✅ Baselines and MITeNAM training functions defined.")

## 7. Run All Experiments

Each model is evaluated under 5-fold stratified cross-validation, repeated with
3 outer seeds (15 fold-level evaluations per model).


In [ ]:
print("="*70)
print("UCI Diabetes 30-day Readmission Experiments")
print("="*70)

print("\n▶ XGBoost...")
res_xgb = run_3seed_5fold(X_uci_full, y_uci, NUM_COLS_UCI, CAT_COLS_UCI, 'uci_xgb', fit_xgb)

print("\n▶ CatBoost...")
res_cat = run_3seed_5fold(X_uci_full, y_uci, NUM_COLS_UCI, CAT_COLS_UCI, 'uci_cat', fit_cat)

print("\n▶ LightGBM...")
res_lgb = run_3seed_5fold(X_uci_full, y_uci, NUM_COLS_UCI, CAT_COLS_UCI, 'uci_lgb', fit_lgb)

print("\n▶ EBM...")
res_ebm = run_3seed_5fold(X_uci_full, y_uci, NUM_COLS_UCI, CAT_COLS_UCI, 'uci_ebm', fit_ebm)

print("\n▶ MITeNAM (cat_heavy)...")
res_mitenam = run_3seed_5fold(X_uci_full, y_uci, NUM_COLS_UCI, CAT_COLS_UCI, 'uci_mitenam', fit_mitenam)

## 8. Final Results and DeLong Statistical Tests

This cell reproduces Table 3 of the paper. F1 is reported at the threshold maximizing
F1 on the OOF predictions; AUROC and AUPRC are threshold-independent.


In [ ]:
# ── Summary table ──
print("="*70)
print("📊 Final Results — UCI Diabetes 30-Day Readmission")
print("="*70)
print(f"\n{'Model':<20}{'AUROC':>10}{'AUPRC':>10}{'F1*':>10}")
print("-"*50)
for name, r in [
    ('XGBoost', res_xgb),
    ('CatBoost', res_cat),
    ('LightGBM', res_lgb),
    ('EBM', res_ebm),
    ('MITeNAM (Ours)', res_mitenam),
]:
    f = r['final']
    print(f"{name:<20}{f['AUROC']:>10.4f}{f['AUPRC']:>10.4f}{f['F1']:>10.4f}")
print("\n* F1 at the threshold maximizing F1 on the OOF predictions.")

# ── DeLong tests: MITeNAM vs each baseline (Real DeLong, not bootstrap) ──
print("\n" + "="*78)
print("🔬 Real DeLong's Test — MITeNAM vs Each Baseline")
print("="*78)
print(f"\n{'Model':<12}{'AUC':>10}{'MITeNAM':>10}{'Δ':>10}{'p (DeLong)':>14}{'Sig':>6}")
print("-"*62)
for name, r in [
    ('XGBoost',  res_xgb),
    ('CatBoost', res_cat),
    ('LightGBM', res_lgb),
    ('EBM',      res_ebm),
]:
    auc_b, auc_m, p = fast_delong(y_uci, r['oof'], res_mitenam['oof'])
    delta = auc_m - auc_b
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    print(f"{name:<12}{auc_b:>10.4f}{auc_m:>10.4f}{delta:>+10.4f}{p:>14.4e}{sig:>6}")

print("\nSignificance: *** p<0.001, ** p<0.01, * p<0.05, ns p>0.05")

## 9. Qualitative Analysis Figure (Figure 2)

This cell trains a single NAM student on fold 1 (seed 42) of the cross-validation
to generate the qualitative analysis figure shown in the paper.

**Figure 2 components:**
- (a) Shape function of the injected teacher signal $p_{teacher}$.
- (b) Per-feature contributions for one positive patient (readmit=1, highest predicted probability).


In [ ]:
# ──────────────────────────────────────────────────────────
# Step 1: Train one NAM student on a single fold for visualization
# ──────────────────────────────────────────────────────────
import xgboost as xgb
from catboost import CatBoostClassifier
import lightgbm as lgb

# First seed (42), first fold
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEEDS[0])
tr_idx, te_idx = next(iter(skf.split(X_uci_full, y_uci)))
X_tr, X_te = X_uci_full.iloc[tr_idx], X_uci_full.iloc[te_idx]
y_tr, y_te = y_uci[tr_idx], y_uci[te_idx]

# Inner train/val split (same as fit_mitenam: 15% stratified, seed 42)
X_tr_in, X_val, y_tr_in, y_val = train_test_split(
    X_tr, y_tr, test_size=0.15, stratify=y_tr, random_state=42)

# Refit scaler for inverse-transform later
imp = SimpleImputer(strategy='median')
_num_tr = imp.fit_transform(X_tr_in[NUM_COLS_UCI])
scaler = StandardScaler().fit(_num_tr)

X_tr_num, X_val_num, X_tr_cat, X_val_cat, cat_dims, X_tr_flat, X_val_flat = \
    prep_data(X_tr_in, X_val, NUM_COLS_UCI, CAT_COLS_UCI)
_, X_te_num, _, X_te_cat, _, _, X_te_flat = prep_data(X_tr_in, X_te, NUM_COLS_UCI, CAT_COLS_UCI)

# Train teachers
spw = (y_tr_in == 0).sum() / max((y_tr_in == 1).sum(), 1)
eps = 1e-7

t_xgb = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    scale_pos_weight=spw, tree_method='hist',
    random_state=42, n_jobs=-1, verbosity=0,
).fit(X_tr_flat, y_tr_in)
t_cat = CatBoostClassifier(
    iterations=300, depth=6, learning_rate=0.05,
    scale_pos_weight=spw, random_state=42, verbose=False,
).fit(X_tr_flat, y_tr_in)
t_lgb = lgb.LGBMClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    scale_pos_weight=spw, objective='binary',
    random_state=42, n_jobs=-1, verbose=-1,
).fit(X_tr_flat, y_tr_in)

def cat_heavy_ensemble(p_x, p_c, p_l):
    return (0.25 * p_x + 0.5 * p_c + 0.25 * p_l).clip(eps, 1 - eps)

p_ens_tr = cat_heavy_ensemble(
    t_xgb.predict_proba(X_tr_flat)[:, 1],
    t_cat.predict_proba(X_tr_flat)[:, 1],
    t_lgb.predict_proba(X_tr_flat)[:, 1],
)
p_ens_val = cat_heavy_ensemble(
    t_xgb.predict_proba(X_val_flat)[:, 1],
    t_cat.predict_proba(X_val_flat)[:, 1],
    t_lgb.predict_proba(X_val_flat)[:, 1],
)
p_ens_te = cat_heavy_ensemble(
    t_xgb.predict_proba(X_te_flat)[:, 1],
    t_cat.predict_proba(X_te_flat)[:, 1],
    t_lgb.predict_proba(X_te_flat)[:, 1],
)
teacher_logit_tr = np.log(p_ens_tr / (1 - p_ens_tr)).astype(np.float32)

# Teacher ensemble AUROC on the test fold (for reference)
ensemble_auroc = roc_auc_score(y_te, p_ens_te)
print(f"Teacher ensemble AUROC (this fold) = {ensemble_auroc:.4f}")

# Append p_teacher to numerical features
X_tr_num_aug = np.concatenate([X_tr_num, p_ens_tr.reshape(-1, 1).astype(np.float32)], axis=1)
X_val_num_aug = np.concatenate([X_val_num, p_ens_val.reshape(-1, 1).astype(np.float32)], axis=1)
X_te_num_aug = np.concatenate([X_te_num, p_ens_te.reshape(-1, 1).astype(np.float32)], axis=1)

# Train one NAM student (seed 42)
model = train_nam(
    X_tr_num_aug, X_tr_cat, y_tr_in,
    X_val_num_aug, X_val_cat, y_val,
    teacher_logit_tr, cat_dims, NAM_SEEDS[0], device,
)
model_cpu = model.cpu().eval()
print("✅ Single NAM trained for visualization.")

In [ ]:
# ──────────────────────────────────────────────────────────
# Step 2: Generate Figure 2 (shape function + patient decomposition)
# ──────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

num_names = list(NUM_COLS_UCI) + ['p_teacher']
cat_names = list(CAT_COLS_UCI)
n_num = len(num_names)
pt_idx = n_num - 1  # p_teacher = last numerical column

# ── (a) Shape function of p_teacher ──
xs = np.linspace(0.0, 1.0, 200).astype(np.float32)
with torch.no_grad():
    fy = model_cpu.num_subnets[pt_idx](torch.FloatTensor(xs).unsqueeze(1)).numpy()

# ── (b) Per-feature contribution for one positive patient ──
with torch.no_grad():
    logits = model_cpu(torch.FloatTensor(X_te_num_aug), torch.LongTensor(X_te_cat)).numpy()
probs = 1 / (1 + np.exp(-logits))
positive_cases = np.where(y_te == 1)[0]
pid = positive_cases[np.argmax(probs[positive_cases])] if len(positive_cases) else int(np.argmax(probs))

with torch.no_grad():
    contribs, labels = [], []
    xn = torch.FloatTensor(X_te_num_aug[pid:pid+1])
    for i in range(n_num):
        c = model_cpu.num_subnets[i](xn[:, i:i+1]).item()
        labels.append('p_teacher' if i == pt_idx else num_names[i])
        contribs.append(c)
    xc = torch.LongTensor(X_te_cat[pid:pid+1])
    for j, (emb, s) in enumerate(zip(model_cpu.cat_embeds, model_cpu.cat_subnets)):
        c = s(emb(xc[:, j])).item()
        contribs.append(c)
        labels.append(cat_names[j])

contribs = np.array(contribs)
order = np.argsort(-np.abs(contribs))[:8]  # top 8 by absolute contribution
top_c = contribs[order][::-1]
top_l = [labels[k] for k in order][::-1]

# ── Plot ──
fig, ax = plt.subplots(1, 2, figsize=(7.0, 2.8))

ax[0].plot(xs, fy, color='#2E5A88', lw=2)
ax[0].axhline(0, color='gray', lw=0.6, ls='--')
ax[0].set_xlabel(r'$p_{\mathrm{teacher}}$ (distilled GBDT signal)')
ax[0].set_ylabel('contribution to log-odds')
ax[0].set_title('(a) Shape function of injected teacher signal', fontsize=9)

cols = ['#C0504D' if v > 0 else '#4F81BD' for v in top_c]
ax[1].barh(range(len(top_c)), top_c, color=cols)
ax[1].set_yticks(range(len(top_c)))
ax[1].set_yticklabels(top_l, fontsize=7)
ax[1].axvline(0, color='gray', lw=0.6)
ax[1].set_xlabel('per-feature contribution (log-odds)')
ax[1].set_title(f'(b) Patient-level decomposition (y=1, p={probs[pid]:.2f})', fontsize=9)

plt.tight_layout()
out_pdf = os.path.join(RESULTS_DIR, 'figure2.pdf')
out_png = os.path.join(RESULTS_DIR, 'figure2.png')
plt.savefig(out_pdf, bbox_inches='tight', dpi=200)
plt.savefig(out_png, bbox_inches='tight', dpi=150)
print(f"💾 Saved: {out_pdf}")
print(f"💾 Saved: {out_png}")
plt.show()